### Matching the name of images to the poverty rate file

In [1]:
from pathlib import Path
import pandas as pd

# =========================
# 1. Load SAIPE poverty file
# =========================
poverty_csv = "all_state_county_poverty.csv"
poverty_df = pd.read_csv(poverty_csv, dtype={"ID": str})

# Keep only needed columns
poverty_df = poverty_df[["ID", "Name", "Percent in Poverty"]].copy()

# Preserve leading zeros
poverty_df["ID"] = poverty_df["ID"].str.zfill(5)

# Remove state rows like 01000, 27000, etc.
poverty_df = poverty_df[~poverty_df["ID"].str.endswith("000")].copy()

# Rename for clarity
poverty_df = poverty_df.rename(columns={
    "ID": "county_id",
    "Name": "county_name_csv",
    "Percent in Poverty": "poverty_rate"
})

# Convert poverty rate to float
poverty_df["poverty_rate"] = poverty_df["poverty_rate"].astype(float)

# =========================

In [ ]:
# -----------------------------
# 2. Read image filenames
# -----------------------------
image_dir = Path(r"C:\Users\zinsc\OneDrive\Desktop\MS DataScience\CSCI 5527 - Deep Learning\Project\Datasets\opt2_county_only\opt2_county_only")

image_records = []

for img_path in image_dir.glob("*"):
    
    if img_path.name.startswith("._"):
        continue

    if img_path.suffix.lower() not in [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".webp"]:
        continue

    stem = img_path.stem
    parts = stem.split("_", 2)

    # expected format: stateID_countyID_countyName
    if len(parts) < 3:
        print("Skipping bad filename:", img_path.name)
        continue

    state_id, county_id, county_name_img = parts

    image_records.append({
        "image_path": str(img_path),
        "image_file": img_path.name,
        "state_id": state_id,
        "county_id": county_id.zfill(5),
        "county_name_img": county_name_img
    })

images_df = pd.DataFrame(image_records)

print("\nimages_df columns:", images_df.columns.tolist())
print("images_df shape:", images_df.shape)
print(images_df.head())

if images_df.empty:
    raise ValueError("No images were found. Check the folder path.")


images_df columns: ['image_path', 'image_file', 'state_id', 'county_id', 'county_name_img']
images_df shape: (3235, 5)
                                          image_path            image_file  \
0  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...  01_01001_Autauga.png   
1  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...  01_01003_Baldwin.png   
2  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...  01_01005_Barbour.png   
3  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...     01_01007_Bibb.png   
4  C:\Users\zinsc\OneDrive\Desktop\MS DataScience...   01_01009_Blount.png   

  state_id county_id county_name_img  
0       01     01001         Autauga  
1       01     01003         Baldwin  
2       01     01005         Barbour  
3       01     01007            Bibb  
4       01     01009          Blount  


In [3]:
# -----------------------------
# 3. Merge images with poverty rates
# -----------------------------
matched_df = images_df.merge(
    poverty_df,
    on="county_id",
    how="left"
)

print("matched_df shape:", matched_df.shape)
print("Matched rows:", matched_df["poverty_rate"].notna().sum())
print("Unmatched rows:", matched_df["poverty_rate"].isna().sum())

print("\nSample matched rows:")
print(
    matched_df[["image_file", "county_id", "county_name_img", "county_name_csv", "poverty_rate"]]
    .head(10)
)

matched_df shape: (3235, 7)
Matched rows: 3109
Unmatched rows: 126

Sample matched rows:
              image_file county_id county_name_img  county_name_csv  \
0   01_01001_Autauga.png     01001         Autauga   Autauga County   
1   01_01003_Baldwin.png     01003         Baldwin   Baldwin County   
2   01_01005_Barbour.png     01005         Barbour   Barbour County   
3      01_01007_Bibb.png     01007            Bibb      Bibb County   
4    01_01009_Blount.png     01009          Blount    Blount County   
5   01_01011_Bullock.png     01011         Bullock   Bullock County   
6    01_01013_Butler.png     01013          Butler    Butler County   
7   01_01015_Calhoun.png     01015         Calhoun   Calhoun County   
8  01_01017_Chambers.png     01017        Chambers  Chambers County   
9  01_01019_Cherokee.png     01019        Cherokee  Cherokee County   

   poverty_rate  
0          12.9  
1          10.5  
2          28.1  
3          21.6  
4          11.7  
5          36.7  
6  

In [4]:
# -----------------------------
# 4. Check matching results
# -----------------------------

# Summary
print("Total county images:", len(images_df))
print("Matched rows:", matched_df["poverty_rate"].notna().sum())
print("Unmatched rows:", matched_df["poverty_rate"].isna().sum())

# Show unmatched rows
unmatched = matched_df[matched_df["poverty_rate"].isna()].copy()

print("\nFirst 20 unmatched rows:")
print(unmatched[["image_file", "county_id", "county_name_img"]].head(20).to_string(index=False))

# Optional: inspect matched rows too
print("\nFirst 10 matched rows:")
print(
    matched_df[matched_df["poverty_rate"].notna()][
        ["image_file", "county_id", "county_name_img", "county_name_csv", "poverty_rate"]
    ].head(10).to_string(index=False)
)

Total county images: 3235
Matched rows: 3109
Unmatched rows: 126

First 20 unmatched rows:
                       image_file county_id      county_name_img
      02_02013_Aleutians_East.png     02013       Aleutians_East
      02_02016_Aleutians_West.png     02016       Aleutians_West
           02_02020_Anchorage.png     02020            Anchorage
              02_02050_Bethel.png     02050               Bethel
         02_02060_Bristol_Bay.png     02060          Bristol_Bay
             02_02063_Chugach.png     02063              Chugach
        02_02066_Copper_River.png     02066         Copper_River
              02_02068_Denali.png     02068               Denali
          02_02070_Dillingham.png     02070           Dillingham
02_02090_Fairbanks_North_Star.png     02090 Fairbanks_North_Star
              02_02100_Haines.png     02100               Haines
       02_02105_Hoonah-Angoon.png     02105        Hoonah-Angoon
              02_02110_Juneau.png     02110               Juneau

In [5]:
final_df = matched_df[matched_df["poverty_rate"].notna()].copy()
final_df.to_csv("county_image_and_poverty_rate.csv", index=False)
print("Saved county_image_and_poverty_rate.csv")

Saved county_image_and_poverty_rate.csv
